# 05 — Self-attention, and why position must be injected

The LSTM (notebook 04) handles order by reading *sequentially*. **Self-attention** is the modern alternative: every position looks at every other position **in parallel**. We solve the same "A before B" order task with a tiny attention model — and, along the way, see a fundamental property: **attention is order-blind unless you add positional information.** (Ties to Part 5 of the curriculum.)

In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

## Same order task as notebook 04

In [ ]:
V, S, A, B = 12, 16, 3, 8
def make_order(n):
    seqs, ys = [], []
    for _ in range(n):
        s = torch.randint(0, V, (S,)); s[s == A] = 0; s[s == B] = 0
        i, j = np.random.choice(S, 2, replace=False)
        s[i] = A; s[j] = B
        seqs.append(s); ys.append(int(i < j))
    return torch.stack(seqs), torch.tensor(ys)

Xtr, ytr = make_order(4000); Xva, yva = make_order(1000)
Xtr, ytr, Xva, yva = (t.to(device) for t in (Xtr, ytr, Xva, yva))

def fit(model, epochs, lr=1e-2, bs=128):
    model = model.to(device); opt = torch.optim.AdamW(model.parameters(), lr=lr); hist = []
    for _ in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr))
        for k in range(0, len(Xtr), bs):
            idx = perm[k:k+bs]
            loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            hist.append((model(Xva).argmax(1) == yva).float().mean().item())
    return hist

## A minimal self-attention classifier

Embed → (optionally add **positional embeddings**) → one `nn.MultiheadAttention` layer (each position attends to all others) → mean-pool → linear. The `use_pos` flag lets us toggle positional information on and off.

In [ ]:
class AttnClassifier(nn.Module):
    def __init__(self, V, D=32, H=4, S=16, use_pos=True):
        super().__init__()
        self.emb = nn.Embedding(V, D)
        self.pos = nn.Embedding(S, D)
        self.use_pos = use_pos
        self.attn = nn.MultiheadAttention(D, H, batch_first=True)
        self.fc = nn.Linear(D, 2)
    def forward(self, x):
        B, S = x.shape
        h = self.emb(x)
        if self.use_pos:
            h = h + self.pos(torch.arange(S, device=x.device))   # inject position
        a, _ = self.attn(h, h, h)      # self-attention: query=key=value=h
        return self.fc(a.mean(dim=1))

## The key experiment: with vs. without positional embeddings

Self-attention treats its input as a **set** — permute the positions and (with mean-pooling) the output is unchanged. So *without* position info it's exactly as order-blind as bag-of-words. Adding a positional embedding per index is what gives it a sense of order.

In [ ]:
no_pos = fit(AttnClassifier(V, use_pos=False), epochs=40)
with_pos = fit(AttnClassifier(V, use_pos=True), epochs=40)
print(f"attention WITHOUT position: {no_pos[-1]:.3f}   <- ~0.5, order-blind")
print(f"attention WITH position:    {with_pos[-1]:.3f}   <- solves the order task")

plt.plot(no_pos, label="no positional emb")
plt.plot(with_pos, label="with positional emb")
plt.axhline(0.5, ls=":", c="gray"); plt.xlabel("epoch"); plt.ylabel("val accuracy")
plt.title("attention needs position to see order"); plt.legend(); plt.show()

That gap is the whole reason Transformers add **positional encodings** (Part 5.3): raw self-attention is permutation-invariant, so order must be injected explicitly. The LSTM got order for free from processing sequentially; attention trades that for parallelism and buys order back with position embeddings.

## LSTM vs. attention — the tradeoff in one line

Both solve the order task. The difference is *how*:
- **LSTM**: sequential — step `t` waits for step `t-1`. Order is implicit.
- **Attention**: parallel — all positions computed at once (great on GPUs), but order-blind, so it needs positional info.

This is exactly the lineage from Part 4 → Part 5: attention replaced recurrence largely because it parallelizes, and positional encodings are the price of giving up sequential processing.

## Takeaways

- **`nn.MultiheadAttention`** (or `F.scaled_dot_product_attention`) lets every position attend to every other, in parallel.
- Self-attention is **permutation-invariant** — order-blind by default; **positional embeddings** inject order (you *saw* the model fail without them).
- Attention vs. recurrence = parallel vs. sequential; same capability here, very different compute profile.

You've now toured the core architectures — linear, MLP, embeddings+pooling, RNN/LSTM, self-attention — each end-to-end with real training and results. From here, the curriculum's Part 5 builds the full Transformer, and the [numpy_pytorch capstone](../numpy_pytorch/05_capstone_transformer_block.ipynb) assembles a tiny GPT.